In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-12-31 09:05:59 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Base clientes con Adquirencia y/o Wompi

## Adquirencia

In [2]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_adqu_vinc

2025-12-31 09:06:46 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
2025-12-31 09:06:46 - [INFO] - Transcurrido: 1767190007, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-12-31 09:06:47 - [INFO] - Finalizo la busqueda, duracion: 00:00.8, resultado: {'year': 2025, 'month': 12, 'day': 21}


{'year': 2025, 'month': 12, 'day': 21}

In [7]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_acept_comer_cli_con_adqui PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_acept_comer_cli_con_adqui STORED AS PARQUET AS WITH outcome1 AS
  (SELECT nit_verificado,
          codigo_unico,
          min(periodo) AS fecha_ym
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR <= """ + str(dict_ult_ing_adqu_vinc['year']) + """
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo IS NOT NULL
   GROUP BY 1,
            2),
                                                                             outcome2 AS
  (SELECT nit_verificado,
          codigo_unico,
          min(fecha_ym) AS fecha_ym
   FROM outcome1
   GROUP BY 1,
            2)
SELECT cast(nit_verificado AS BIGINT) AS num_doc,
       'NIT' AS cod_tipo_doc,
       cast(codigo_unico AS BIGINT) AS codigo_unico,
       'adqui' AS producto
FROM outcome2;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_acept_comer_cli_con_adqui;"""
helper.ejecutar_consulta(sql_compute)

------------------------------------------------------------------------------------------
  i   tipo                  nombre                    estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 5/5    DROP proceso.mdo_acept_comer_cli_con_adqui   finalizado   09:29:10 AM     00:00.1 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------
  i   tipo                  nombre                    estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 6/6  CREATE proceso.mdo_acept_comer_cli_con_adqui   finalizado   09:29:10 AM     00:00.4 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------

## Wompi

In [8]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch


2025-12-31 09:29:14 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-12-31 09:29:15 - [INFO] - Finalizo la busqueda, duracion: 00:00.3, resultado: {'year': 2025, 'month': 12, 'day': 30}


{'year': 2025, 'month': 12, 'day': 30}

In [9]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_acept_comer_cli_con_wompi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """CREATE TABLE proceso.mdo_acept_comer_cli_con_wompi STORED AS PARQUET AS
SELECT cast(documento_identidad AS BIGINT) AS num_doc,
       tipo_documento AS cod_tipo_doc,
       cast(id_comercio AS BIGINT) AS codigo_unico,
       'wompi' AS producto
FROM resultados_wompi.wompi_merchants
WHERE YEAR = 2025
  AND MONTH = 12
  AND DAY = 30
  AND modelo = 'Agregador'
  AND activo = 'A'
  AND desembolsos_permitidos = 'Si';"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_acept_comer_cli_con_wompi;"""
helper.ejecutar_consulta(sql_compute)

------------------------------------------------------------------------------------------
  i   tipo                  nombre                    estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 8/8    DROP proceso.mdo_acept_comer_cli_con_wompi   finalizado   09:30:31 AM     00:00.3 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------
  i   tipo                  nombre                    estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 9/9  CREATE proceso.mdo_acept_comer_cli_con_wompi   finalizado   09:30:31 AM     00:00.3 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------

# Crear tabla clientes con aceptación comercios [Adquirencia y/o Wompi]

In [13]:
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_acept_comer_cli_con_adqui_o_wompi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """CREATE TABLE proceso_vdm.mdo_acept_comer_cli_con_adqui_o_wompi STORED AS PARQUET AS
SELECT num_doc,
       cod_tipo_doc,
       codigo_unico,
       producto
FROM proceso.mdo_acept_comer_cli_con_adqui
UNION ALL
SELECT num_doc,
       cod_tipo_doc,
       codigo_unico,
       producto
FROM proceso.mdo_acept_comer_cli_con_wompi;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_acept_comer_cli_con_adqui_o_wompi;"""
helper.ejecutar_consulta(sql_compute)


-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 17/17    DROP ...mdo_acept_comer_cli_con_adqui_o_wompi   finalizado   09:34:56 AM     00:00.0 
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 18/18  CREATE ...mdo_acept_comer_cli_con_adqui_o_wompi   finalizado   09:34:56 AM     00:00.5 
-----------------------------------------------------------------------------------------------
----------------------------------------